# Métodos de Jacobi

Dada una matriz simétrica $A$ de tamaño $n$, la idea es llevar a $A$ lo más cerca posible a una forma diagonal. Si denotamos por $D^{(0)}$ la diagonal de $A$ y $N^{(0)}$ la parte no diagonal, entonces

$$\begin{align}
A&=D^{(0)}+N^{(0)}\\
||A||_F^2&=||D^{(0)}||_F^2+||N^{(0)}||_F^2
\end{align}$$

Se quiere un algoritmo que en cada iteración disminuya la norma de $N^{(i)}$:

$$||N^{(i+1)}||_F<||N^{(i)}||_F$$

Definimimos 

$$\text{off}(A) = ||N^{(0)}||_F$$

Para esto se utilizan matrices de Givens para eliminar selectivamente componentes de $$N^{(i)}$$. Se escogen índices $p,q$ con $p<q$ y se busca eliminar dos entradas de la matriz por fuera de la diagonal, es decir, queremos escoger $c$ y $s$ adecuados para que la siguiente matriz sea diagonal:

$$
\begin{align}
\left[\begin{array}{cc}
b_{pp}& b_{pq}\\
b_{qp}& b_{qq}
\end{array}\right] &= 
\left[\begin{array}{cc}
c& s\\
-s& c
\end{array}\right]^T
\left[\begin{array}{cc}
a_{pp}& a_{pq}\\
a_{qp}& a_{qq}
\end{array}\right]
\left[\begin{array}{cc}
c& s\\
-s& c
\end{array}\right]\\
 \left[\begin{array}{cc}
b_{pp}& 0\\
0& b_{qq}
\end{array}\right] &=
\end{align}
$$

Recordando que las transformaciones ortogonales preservan la norma de Frobenius, tenemos 

$$a_{pp}^2+a_{qq}^2 + 2a_{pq}^2 = b_{pp}^2 + b_{qq}^2$$

Notando $B=G(p,q,\theta)^TAG(p,q,\theta)$, dado que las diagonales de $B$ y $A$ solo difieren en las componentes con índices $pp$ y $qq$, tenemos que

$$\begin{align}
\text{off}(B)^2 &= ||B||_F^2 - \sum_{i=1}^{n} b_{ii}^2 \\
&= ||A||_F^2 - \sum_{i=1}^{n} a_{ii}^2 + (a_{pp}^2+a_{qq}^2 - b_{pp}^2- b_{qq}^2)\\
&=\text{off}(A)^2-2a_{pq}^2
\end{align}$$

es decir, 

$$\text{off}(B)\leq \text{off}(A)$$

por lo que $B$ es "más diagonal" que $A$. Para escoger $c$ y $s$ adecuados, vemos que 

$$0= a_{pq}(c^2-s^2) + (a_{pp}-a_{qq})cs$$

Asumimos que $a_{pq}\neq 0$, o de lo contrario, escogemos la matriz de Givens igual a la identidad. Tenemos 

$$\begin{align}
\frac{a_{qq}-a_{pp}}{2 a_{pq}} &= \frac{(c^2-s^2)}{2 cs}\\
&= \frac{1}{2} \left(\frac{c}{s}-\frac{s}{c}\right)
\end{align}$$

escogiendo $\tau = \frac{a_{qq}-a_{pp}}{2 a_{pq}}$ y $t = \frac{s}{c}$, lo anterior se escribe

$$
\begin{align}
\tau &= \frac{1}{2}\left(\frac{1}{t}-t\right)\\
2t\tau&= 1-t^2\\
t^2+2t\tau-1&=0
\end{align}
$$

Las raices son 

$$
t=-\tau \pm \sqrt{1+\tau^2}
$$

y se despeja $c$ y $s$ en términos de $t$ así,

$$c = \frac{1}{\sqrt{1+t^2}} \quad s = tc$$

se escoge la menor de las raíces.

La función **symSchur2** toma una matriz simétrica $A$ y dos índices $p,q$ y devuelve $c$ y $s$ tales que 

$$\left[\begin{array}{cc}
c& s\\
-s& c
\end{array}\right]^T
\left[\begin{array}{cc}
a_{pp}& a_{pq}\\
a_{qp}& a_{qq}
\end{array}\right]
\left[\begin{array}{cc}
c& s\\
-s& c
\end{array}\right]$$
es diagonal.

In [26]:
using LinearAlgebra

In [1]:
function symSchur2(A, p ,q)
    if A[p,q] != 0
        τ = (A[q,q] - A[p,p])/(2*A[p,q])
        if τ >= 0
            t = 1/(τ + sqrt(1+τ^2))
        else
            t = -1/(-τ + sqrt(1+τ^2))
        end
        c = 1/sqrt(1 + t^2)
        s = t*c 
    else
        c = 1
        s = 0
    end
    return c, s
end

symSchur2 (generic function with 1 method)

Probamos con una matriz $2\times 2$:

In [15]:
B = [1 -2
     -2 1]
c, s = symSchur2(B,1,2)

print("La matriz G'BG es:")
display([c -s; s c]*B*[c s; -s c])

La matriz G'BG es:

2×2 Matrix{Float64}:
 3.0   0.0
 0.0  -1.0

Probamos con una matriz más grande:

In [23]:
B = 
[8.0  3.0  4.0  1.0
 3.0  9.0  4.0  1.0
 4.0  4.0  8.0  5.0
 1.0  1.0  5.0  8.0]

c, s = symSchur2(B,2,4)


C = copy(B)
C[[2, 4],:] = [c -s;s c]*C[[2, 4],:]
C[:,[2,4]] = C[:,[2,4]]*[c s;-s c]

print("La matriz G'BG es:")
display(C) 


La matriz G'BG es:

4×4 Matrix{Float64}:
  8.0        3.07768     4.0      -0.726543
  3.07768    9.61803     6.03126  -3.11804e-16
  4.0        6.03126     8.0       2.15033
 -0.726543  -3.8783e-16  2.15033   7.38197

In [24]:
B

4×4 Matrix{Float64}:
 8.0  3.0  4.0  1.0
 3.0  9.0  4.0  1.0
 4.0  4.0  8.0  5.0
 1.0  1.0  5.0  8.0

In [31]:
function maxpq(A)
    n = size(A)[1]
    p = 1
    q = 2
    for i = 1:n
        for j = i:n
            if i != j
                if abs(A[i,j]) > abs(A[p,q])
                    p = i
                    q = j
                end
            end
        end
    end
    return p,q
end

function offdiagonal(A)
    return sqrt(norm(A ,2)^2 - norm(diag(A))^2)
end

function classicalJacobi(A, tol, iteraciones)
    n = size(A)[1]
    V = UniformScaling(n)
    eps = tol*norm(A, 2)
    for k = 1:iteraciones
        p, q = maxpq(A)
        c, s = symSchur2(A,p,q)
        A[[p, q],:] = [c -s;s c]*A[[p, q],:]
        A[:,[p,q]] = A[:,[p,q]]*[c s;-s c]
        if offdiagonal(A) <= eps
            return A
        end
    end
    print("máximo de iteraciones")
end

classicalJacobi (generic function with 1 method)

In [33]:
classicalJacobi(B, 0.00001, 1000)

4×4 Matrix{Float64}:
  8.16528       1.18011e-7   -8.52038e-5   -2.4883e-12
  1.18011e-7    5.46082      -1.56586e-12  -4.25658e-20
 -8.52038e-5   -1.56603e-12   1.78652      -7.42632e-8
 -2.48814e-12  -8.34983e-17  -7.42632e-8   17.5874